# init and Helpers

In [ ]:
# Load variables from .env file (keeps secret key in .env since .env is not pushed into vrs control tools)
from dotenv import load_dotenv
load_dotenv()

# Create Anthropic Client
from anthropic import Anthropic
client = Anthropic()
model="claude-sonnet-4-0" # can be any model

In [ ]:
# Helpers
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
    model=model,
    max_tokens=1000, # Token limit per message, can be changed
    messages=messages, # Stateful
    )
    return message.content[0].text

# Tool Function

In [ ]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_fromat cannot be empty") # return error so that claude can retry the tool call properly
    return datetime.now().strftime(date_format)

get_current_datetime() # gives you this format "%Y-%m-%d %H:%M:%S

# JSON Schema

- Allows claude to understand what arguments are needed. It's not an LLM thing.
- The schema contains a tool name and description to help claude know when to call it and what for. After the name and desc, it will then contain the actual input_schema
- Just ask Claude to generate valid schema with claude's best practices in their documentation


In [ ]:
# A good practice is to add '_schema' to the end of the tool's name

from anthropic.types import ToolParam # Optional, but typically used to prevent type errors 

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

# ToolUseBlock

- usually only have either `user` msg or `assistant` msg
- with tools, it now has a `ToolUseBlock` (it is the request to call a tool)



In [ ]:
messages = []

# Basically add_user_message function
messages.append(
    {
        "role": "user",
        "content": "What is the exact time in HH:MM::SS?"
    }
)

# Basically chat() function
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

# Basically add_assistant_message function 
messages.append(
    {
    "role": "assistant",
    "content": response.content # Basically we are appending the output (aka assistant) of the ToolUseBlock to the messages[]
    }
)

## Just showing how to fix typeError from tool calls

In [ ]:
# The ** fixes the Type Error because the response.content (output) gave a dictionary 'date_format': %H:%M:%S instead of just %H:%M:%S
result = get_current_datetime(**response.content[1].input) 

# ToolResultsBlock

Flow of messages from user to assistant:
- user gives query
- assistant gives answer + ToolUseBlock
- user gives back ToolResultsBlock

1) `tool_use_id` - for claude to know which tool results are for which requests, because when the results come back, they might not be ordered
2) `type` - type of the block, which is tool_result
3) `content` - the output
4) `is_error` - True if error occured, False if not

In [ ]:
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": response.content[1].id   # Just tells you where to find the id from the response output 
                "content": result
                "is_error": False
            }
        ]
    }
)

# Basically chat() function
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

# Refactoring helpers for multiple turns

In [ ]:
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user", 
        "content": message.content if isinstance(message, Message) else message, # Checks if message is a anthropic Message object, if yes use message.content
        }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant", 
        "content": message.content if isinstance(message, Message) else message,
        }
    messages.append(assistant_message)

In [ ]:
messages = []

add_user_message(messages, "What is the exact time in HH:MM::SS?")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

add_assistant_message(messages, response)

In [ ]:
def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_token": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequence": stop_sequences,
    }

    if tools:
        params["tools"] = tools
    
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message # not more content[0] because there are multi blocks now

# Extract all text from message block, and add it to a list
def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

# Implementation of multiple turns with error handling

In [ ]:
# To support multiple tools
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "another_tool":
        return another_tool(**tool_input)
    


def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"  # Basically filtering for only tool_use blocks
    ]
    tool_result_blocks = []
    
    for tool_request in tool_requests:
        tool_output = run_tool(tool_request.name, tool_request.input)
        try:
            tool_output = get_current_datetime(**tool_request.input)
            tool_result_blocks = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            }
        # Error Handling used to tell you what is the exact error
        except Exception as e:
            tool_result_blocks = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True
            }

        tool_result_blocks.append(tool_result_blocks)

    return tool_result_blocks

In [ ]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))
        
        if response.stop_reason != "tool_use":  # if claude doesnt want to use tools anymore, end loop
            break
            
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    
    return messages